<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/01-arbol-de-decision/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 2.1 · Árbol de decisión: tu `=SI()` anidado, pero aprendido

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas tres celdas. Dejan listas `despachos`, `entreno`, `prueba`, `x_entreno` y `x_prueba`. Deben quedar `30516` y `11321` filas.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def preparar_despachos(despachos):
    """Deja los datos que se conocen al preparar un despacho: textos como 0 y 1, mes y día."""
    fecha = pd.to_datetime(despachos["fecha"])
    x = pd.get_dummies(despachos[["ciudad", "tipo", "transportista"]], dtype=int)
    x = x.drop(columns=["ciudad_Lima", "tipo_Bodega", "transportista_Propio"], errors="ignore")
    x["mes"] = fecha.dt.month
    x["dia"] = fecha.dt.dayofweek
    return x.join(despachos[["distancia_km", "lineas", "presentaciones"]])

In [3]:
despachos = pd.read_csv(DATOS + "despachos.csv")

entreno = despachos[despachos["fecha"] < "2025-07-01"]
prueba = despachos[despachos["fecha"] >= "2025-07-01"]

x_entreno = preparar_despachos(entreno)
x_prueba = preparar_despachos(prueba)

print(len(entreno), len(prueba))

30516 11321


### 1. ¿Fuga o no?

Responde en un comentario si cada dato se puede usar para predecir `llego_tarde` en un despacho que está por salir, y por qué:

- a) `distancia_km`
- b) `transportista`
- c) Un ticket de queja que el cliente escribió sobre ese despacho

In [4]:
# a) Sí: la distancia del cliente se conoce desde antes.
# b) Sí, si ya está asignado al preparar el despacho.
# c) No: es fuga. El ticket se escribe después de que el despacho llegó tarde.

### 2. Tu árbol

Entrena un árbol de profundidad 2 y guárdalo en `arbol`. Usa `random_state=0`.

In [5]:
arbol = DecisionTreeClassifier(max_depth=2, random_state=0)
arbol.fit(x_entreno, entreno["llego_tarde"])

print(arbol.get_n_leaves())

4


### 3. Léelo

Muestra las reglas con `print(export_text(arbol, feature_names=list(x_entreno.columns), show_weights=True))`. En cada hoja, `weights` da cuántos despachos llegaron a tiempo y cuántos tarde. Responde en un comentario: ¿cuál es la primera pregunta del árbol? ¿Qué hoja tiene el mayor porcentaje de tardíos?

In [6]:
print(export_text(arbol, feature_names=list(x_entreno.columns), show_weights=True))

# La primera pregunta es si `dia` es menor o igual que 4.5: de lunes a viernes.
# La de mayor riesgo es `dia > 4.5` con `transportista_Tercero > 0.5`: fin de semana con tercero.

|--- dia <= 4.50
|   |--- ciudad_Cusco <= 0.50
|   |   |--- weights: [19606.00, 1026.00] class: 0
|   |--- ciudad_Cusco >  0.50
|   |   |--- weights: [2208.00, 372.00] class: 0
|--- dia >  4.50
|   |--- transportista_Tercero <= 0.50
|   |   |--- weights: [4608.00, 257.00] class: 0
|   |--- transportista_Tercero >  0.50
|   |   |--- weights: [1528.00, 911.00] class: 0



### 4. Riesgo por hoja

Calcula, en `prueba`, cuántos despachos caen en cada hoja y qué parte llegó tarde, como en la clase. La hoja de mayor riesgo debe tener `875` despachos.

In [7]:
riesgo = pd.Series(arbol.predict_proba(x_prueba)[:, 1], index=prueba.index).round(3)

prueba.groupby(riesgo)["llego_tarde"].agg(["size", "mean"]).sort_index(ascending=False).round(3)

,size,mean
0.374,875,0.387
0.144,1008,0.127
0.053,1722,0.060
0.050,7716,0.059


### 5. De hoja a Excel

Mira la hoja de mayor riesgo. Escribe en un comentario la fórmula `=SI()` (`IF`) que la describa, suponiendo que `dia` está en la columna B y `transportista_Tercero` en la columna C.

In [8]:
# La hoja de mayor riesgo es fin de semana con transportista tercero.
# En inglés, IF y AND:
# =SI(Y(B2>4.5, C2>0.5), "Vigilar", "OK")

### 6. ¿Lo usarías así?

Responde en un comentario. El árbol dice "A tiempo" en todas las hojas. Tu jefe concluye: *"entonces no hay que vigilar nada"*. ¿Qué le dirías?

In [9]:
# Que "A tiempo" solo significa que ninguna hoja pasa del 50 %. Las hojas son muy distintas:
# el riesgo cambia mucho de una a otra. Hay que vigilar primero las de mayor riesgo.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 2.2: Cuando el árbol se aprende todo**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0